# ModelEdge — QLoRA Fine-tuning on Kaggle 2x T4

**Kaggle setup before running:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Secrets → add key `WANDB_API_KEY` with your W&B token
3. Run all cells top-to-bottom

Training uses `accelerate launch --num_processes 2` so both T4s are active.
Adapter weights are saved to `/kaggle/working/outputs/finetuned`.

In [ ]:
# Verify both GPUs are visible
!nvidia-smi

In [ ]:
!pip install -q unsloth[colab-new] peft accelerate datasets bitsandbytes trl wandb
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
# Load W&B key from Kaggle Secrets → no hardcoded tokens
import os
from kaggle_secrets import UserSecretsClient

os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

import wandb
wandb.login()

In [ ]:
# Clone the repo
!git clone https://github.com/sakshiasati17/ModelEdge.git /kaggle/working/ModelEdge
%cd /kaggle/working/ModelEdge

# Point config output_dir at Kaggle working dir
import re, pathlib
cfg_path = pathlib.Path('training/config.yaml')
cfg_path.write_text(
    re.sub(r'output_dir:.*', 'output_dir: /kaggle/working/outputs/finetuned',
           cfg_path.read_text())
)

In [ ]:
# Prepare MedQA dataset
!python data/prepare_dataset.py --dataset medqa --output data/processed/

In [ ]:
# Inspect a few training samples
import json

with open('data/processed/medqa/train.jsonl') as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        print(f'--- Sample {i+1} ---')
        print(rec['text'][:400])
        print()
        if i >= 2:
            break

In [ ]:
# Write Accelerate config for 2x T4 (multi-GPU, no TPU)
accel_config = """
compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
num_machines: 1
num_processes: 2
gpu_ids: all
mixed_precision: fp16
main_process_port: 29500
"""
import pathlib
pathlib.Path('/root/.cache/huggingface/accelerate/default_config.yaml').parent.mkdir(parents=True, exist_ok=True)
pathlib.Path('/root/.cache/huggingface/accelerate/default_config.yaml').write_text(accel_config.strip())
print('Accelerate config written.')

In [ ]:
# Launch fine-tuning on both T4s.
# accelerate.prepare() inside finetune.py wraps:
#   model, optimizer, train_dataloader, val_dataloader, lr_scheduler
!accelerate launch --num_processes 2 training/finetune.py --config training/config.yaml

In [ ]:
# Verify adapter files were saved
!ls -lh /kaggle/working/outputs/finetuned/

In [ ]:
# Quick inference test on GPU 0
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    '/kaggle/working/outputs/finetuned',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

test_prompt = (
    'Below is a medical question. Answer it accurately and concisely.\n\n'
    '### Instruction:\nWhat is the first-line treatment for hypertension?\n\n'
    '### Input:\n\n'
    '### Response:\n'
)
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# Serve the fine-tuned model with vLLM across both T4s
# Run this in a Kaggle terminal tab (Add-ons → Terminal)
print('Open a terminal and run:')
print()
print('  python -m vllm.entrypoints.openai.api_server \\')
print('    --model /kaggle/working/outputs/finetuned \\')
print('    --tensor-parallel-size 2 \\')
print('    --host 0.0.0.0 --port 8001')